# 06. Recording-level leakage and condition shift on D3 (E5)

Unit: 1 s segment (30 per recording; 3,600 total). Model: GBDT.

- Part A: v1 70/30 policies, 10 seeds, all split attempts logged
- Part B: out-of-fold (OOF) design, all segments predicted once per policy
- Part C: int16 vs. PCM_32 decoding, identical splits

| Policy | Folds | Held out |
|---|---|---|
| S1 segment-random | 5, stratified | – |
| S2 recording-independent | 5, stratified group | recording (one hydrophone) |
| S2b run-independent | 5, stratified group | physical run (H1 + H2) |
| S3 operating condition (LOCO) | 6, leave-one-group-out | flow × noise |
| S3′ topology (LOTO) | 2, leave-one-group-out | pipe topology |

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import src as la
from src import config as cfg, io_utils, features, evaluation as ev, sfb, diagnostics as dg, reporting as rp
cfg.apply_mpl_style(); cfg.banner()
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit

M3 = io_utils.scan_d3()
H = M3[(~M3.is_bg) & (M3.modality == "hydrophone") & M3.label.notna()].reset_index(drop=True)
MAX_SEG = 30     # 30 s clip = 30 x 1 s segments

LeakAudit | root=C:\Users\ecredible\Desktop\LeakAudit_revision1\LeakAudit | mode=FULL | seeds=10 | bootstrap=2000


## 1. 1 s segments (PCM_32 and int16 decoding)

Band-pass filter → 1 s segments → summary features; cached with file IDs

In [2]:
def build_segments(decoding="pcm32", win_sec=cfg.WIN_SEC, max_seg=MAX_SEG):
    name = f"d3_seg_hydrophone_{win_sec}_{decoding}"
    c = io_utils.load_cache(name, H.file_id, by_unique=True)
    if c is None:
        Xs, ids, segs = [], [], []
        for i, r in tqdm(H.iterrows(), total=len(H), desc=f"segments/{decoding}"):
            x, sr = io_utils.load_d3_signal(r.to_dict(), decoding=decoding)
            xb = features.bandpass(x, sr)
            n = int(win_sec * sr); k = min(max_seg, len(xb) // n)
            for j in range(k):
                Xs.append(features.summary_features(xb[j*n:(j+1)*n], sr))
                ids.append(r.file_id); segs.append(j)
        io_utils.save_cache(name, np.vstack(Xs), ids, seg=np.array(segs))
        c = io_utils.load_cache(name)
    meta = H.set_index("file_id").loc[c["ids"].astype(str)].reset_index()
    meta["seg"] = c["seg"]
    return c["X"], meta

Xseg, S = build_segments("pcm32")
Xleg, S_leg = build_segments("int16_legacy")
assert (S.file_id.values == S_leg.file_id.values).all()
y = S.label.to_numpy()
print(f"segments: {Xseg.shape} from {S.file_id.nunique()} recordings, {S.run_id.nunique()} runs")
print(S.label.value_counts().to_string())

segments/pcm32:   0%|          | 0/120 [00:00<?, ?it/s]

segments/int16_legacy:   0%|          | 0/120 [00:00<?, ?it/s]

segments: (3600, 199) from 120 recordings, 60 runs
label
leak      2880
noleak     720


## 2. Part A: v1 policies (70/30, 10 seeds)

- S1 segment-random / S2 recording-independent / S3 v1 condition (topology × leak type)
- Split attempts without both classes: logged and excluded
- Steps in accuracy, BA and excess over majority rate

In [3]:
def v1_policies(X, decoding):
    rows, attempts = [], []
    rec, cond = S.file_id.to_numpy(), S.cond_topo_leak.to_numpy()
    for seed in range(cfg.N_SEEDS):
        cands = [("S1_segment_random", train_test_split(np.arange(len(y)), test_size=.3,
                                                        random_state=seed, stratify=y)),
                 ("S2_recording_indep", next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                                              .split(np.arange(len(y)), y, groups=rec))),
                 ("S3_condition_indep_v1", next(GroupShuffleSplit(1, test_size=.3, random_state=seed)
                                                 .split(np.arange(len(y)), y, groups=cond)))]
        for proto, (a, b) in cands:
            ok_tr, ok_te = len(np.unique(y[a])) > 1, len(np.unique(y[b])) > 1
            attempts.append(dict(decoding=decoding, protocol=proto, seed=seed,
                                 held_out=",".join(sorted(set(cond[b]))) if proto.startswith("S3") else "",
                                 n_train=len(a), n_test=len(b), usable=ok_tr and ok_te,
                                 reason="" if (ok_tr and ok_te) else
                                 ("no no-leak segment in test" if not ok_te else "no no-leak segment in train")))
            if ok_tr and ok_te:
                r = ev.evaluate(X[a], y[a], X[b], y[b], "gbdt", seed)
                rows.append(dict(decoding=decoding, protocol=proto, seed=seed,
                                 **{k: r[k] for k in ["acc", "bacc", "mf1", "base", "excess",
                                                      "rec_leak", "rec_noleak"]}))
    return pd.DataFrame(rows), pd.DataFrame(attempts)

e5, att = v1_policies(Xseg, "pcm32")
e5_leg, att_leg = v1_policies(Xleg, "int16_legacy")
pd.concat([att, att_leg]).to_csv(cfg.RESULTS / "E5_split_attempts.csv", index=False)
e5.to_csv(cfg.RESULTS / "E5_recording_leakage_raw.csv", index=False)
s5 = e5.groupby("protocol").agg(acc=("acc", "mean"), sd=("acc", "std"), bacc=("bacc", "mean"),
                                base=("base", "mean"), excess=("excess", "mean"),
                                rec_noleak=("rec_noleak", "mean"), n=("acc", "size")).reset_index()
s5.to_csv(cfg.RESULTS / "E5_recording_leakage_summary.csv", index=False)
print(s5.round(4).to_string(index=False))
print("\nattempted condition splits (v1):")
print(att[att.protocol == "S3_condition_indep_v1"][["seed", "held_out", "usable", "reason"]].to_string(index=False))

g = s5.set_index("protocol")
steps_v1 = []
for pa, pb, meaning in [("S1_segment_random", "S2_recording_indep", "recording-level sharing"),
                        ("S2_recording_indep", "S3_condition_indep_v1", "unseen topology x leak type")]:
    steps_v1.append(dict(step=f"{pa[:2]}->{pb[:2]}", meaning=meaning,
                         drop_acc=g.loc[pa, "acc"] - g.loc[pb, "acc"],
                         drop_excess=g.loc[pa, "excess"] - g.loc[pb, "excess"],
                         drop_bacc=g.loc[pa, "bacc"] - g.loc[pb, "bacc"],
                         prior_shift=g.loc[pa, "base"] - g.loc[pb, "base"]))
steps_v1 = pd.DataFrame(steps_v1)
steps_v1.to_csv(cfg.RESULTS / "E5_v1_steps_excess.csv", index=False)
print("\nv1 steps in raw accuracy and in excess over the same-fold majority rate (R1-Major 3):")
print(steps_v1.round(4).to_string(index=False))
for _, r in steps_v1.iterrows():
    rp.record(f"D3.v1.{r.step}.drop_acc", r.drop_acc, "E5_v1_steps_excess.csv", "R1-M3")
    rp.record(f"D3.v1.{r.step}.drop_excess", r.drop_excess, "E5_v1_steps_excess.csv", "R1-M3")
for p, r in g.iterrows():
    rp.record(f"D3.v1.{p}.acc", r.acc, "E5_recording_leakage_summary.csv", "R1-M7",
              note=f"bacc={r.bacc:.4f}, base={r.base:.4f}, usable seeds={int(r.n)}")

             protocol    acc     sd   bacc   base  excess  rec_noleak  n
    S1_segment_random 0.9874 0.0032 0.9737 0.8000  0.1874      0.9509 10
   S2_recording_indep 0.9052 0.0467 0.7952 0.8056  0.0996      0.6075 10
S3_condition_indep_v1 0.5950 0.1102 0.4489 0.6667 -0.0717      0.0106  5

attempted condition splits (v1):
 seed          held_out  usable                      reason
    0 BR_LC,BR_OL,LO_NL    True                            
    1 BR_LC,LO_GL,LO_OL   False  no no-leak segment in test
    2 BR_GL,BR_OL,LO_CC   False  no no-leak segment in test
    3 BR_GL,BR_OL,LO_CC   False  no no-leak segment in test
    4 BR_NL,BR_OL,LO_NL   False no no-leak segment in train
    5 BR_LC,LO_CC,LO_OL   False  no no-leak segment in test
    6 BR_GL,LO_LC,LO_NL    True                            
    7 BR_CC,LO_CC,LO_NL    True                            
    8 LO_GL,LO_NL,LO_OL    True                            
    9 BR_OL,LO_LC,LO_NL    True                            

v1 steps in r

## 3. Part B: out-of-fold design

### 3.1 Fold contents (LOCO, LOTO)

Held-out group, train/test sizes, classes, topologies, operating conditions and leak types in training

In [4]:
POLICIES = {
    "S1_segment_random":  ("stratified", None, True),
    "S2_recording_indep": ("stratified_group", S.file_id.to_numpy(), True),
    "S2b_run_indep":      ("stratified_group", S.run_id.to_numpy(), True),
    "S3_opcond_LOCO":     ("logo", S.op_condition.to_numpy(), False),
    "S3_topology_LOTO":   ("logo", S.topology.to_numpy(), False),
}
fold_desc = []
for name in ["S3_opcond_LOCO", "S3_topology_LOTO"]:
    scheme, grp, _ = POLICIES[name]
    for k, (tr, te) in enumerate(ev.make_folds(scheme, y, grp)):
        fold_desc.append(dict(policy=name, fold=k, held_out=",".join(sorted(set(grp[te]))),
                              n_train=len(tr), n_test=len(te),
                              train_classes=dict(pd.Series(y[tr]).value_counts()),
                              test_classes=dict(pd.Series(y[te]).value_counts()),
                              train_topologies=",".join(sorted(set(S.topology[tr]))),
                              train_opconds=len(set(S.op_condition[tr])),
                              train_leaktypes=",".join(sorted(set(S.leaktype[tr])))))
fold_desc = pd.DataFrame(fold_desc)
fold_desc.to_csv(cfg.RESULTS / "E5_condition_fold_contents.csv", index=False)
print(fold_desc.to_string(index=False))

          policy  fold     held_out  n_train  n_test                 train_classes                  test_classes train_topologies  train_opconds train_leaktypes
  S3_opcond_LOCO     0       0.18_N     3000     600 {'leak': 2400, 'noleak': 600}  {'leak': 480, 'noleak': 120}            BR,LO              5  CC,GL,LC,NL,OL
  S3_opcond_LOCO     1       0.47_N     3000     600 {'leak': 2400, 'noleak': 600}  {'leak': 480, 'noleak': 120}            BR,LO              5  CC,GL,LC,NL,OL
  S3_opcond_LOCO     2         ND_N     3000     600 {'leak': 2400, 'noleak': 600}  {'leak': 480, 'noleak': 120}            BR,LO              5  CC,GL,LC,NL,OL
  S3_opcond_LOCO     3        ND_NN     3000     600 {'leak': 2400, 'noleak': 600}  {'leak': 480, 'noleak': 120}            BR,LO              5  CC,GL,LC,NL,OL
  S3_opcond_LOCO     4  TRANSIENT_N     3000     600 {'leak': 2400, 'noleak': 600}  {'leak': 480, 'noleak': 120}            BR,LO              5  CC,GL,LC,NL,OL
  S3_opcond_LOCO     5 TRANSIENT_N

### 3.2 OOF accuracy and uncertainty by policy

- Point estimate: fold seed 0
- Uncertainty: model initialisation SD, split SD (10 seeds; randomised folds only), run-level bootstrap

In [5]:
def oof_all(X, decoding):
    out, preds = [], {}
    for name, (scheme, grp, randomised) in POLICIES.items():
        fold_fn = (lambda s, sc=scheme, gr=grp: ev.make_folds(sc, y, gr, 5, seed=s))
        unc, ok0 = dg.uncertainty_decomposition(X, y, S.run_id.to_numpy(), fold_fn, "gbdt",
                                                n_rep=cfg.N_SEEDS, n_boot=cfg.N_BOOT,
                                                randomised_folds=randomised)
        folds = fold_fn(0)
        pred, _ = ev.oof_predict(X, y, folds, "gbdt", 0)
        s = ev.score(y, pred)
        preds[name] = (pred, ev.fold_majority_rule(y, folds))
        out.append(dict(decoding=decoding, policy=name, acc=s["acc"], bacc=s["bacc"], mf1=s["mf1"],
                        rec_leak=s["rec_leak"], rec_noleak=s["rec_noleak"],
                        rule=float(preds[name][1].mean()), excess=s["acc"] - preds[name][1].mean(),
                        init_sd=unc["init_sd"], split_sd=unc["split_sd"],
                        run_boot_lo=unc["group_lo"], run_boot_hi=unc["group_hi"],
                        excess_lo=unc["excess_lo"], excess_hi=unc["excess_hi"],
                        n_runs=unc["n_clusters"], cm=s["cm"]))
    return pd.DataFrame(out), preds

oof, preds = oof_all(Xseg, "pcm32")
oof.to_csv(cfg.RESULTS / "E5_oof_policies.csv", index=False)
print(oof.drop(columns="cm").round(4).to_string(index=False))

decoding             policy    acc   bacc    mf1  rec_leak  rec_noleak  rule  excess  init_sd  split_sd  run_boot_lo  run_boot_hi  excess_lo  excess_hi  n_runs
   pcm32  S1_segment_random 0.9872 0.9748 0.9798    0.9955      0.9542   0.8  0.1872      0.0    0.0010       0.9783       0.9939     0.0930     0.2842      60
   pcm32 S2_recording_indep 0.9267 0.8469 0.8755    0.9799      0.7139   0.8  0.1267      0.0    0.0077       0.8881       0.9600     0.0550     0.2047      60
   pcm32      S2b_run_indep 0.9358 0.8604 0.8909    0.9861      0.7347   0.8  0.1358      0.0    0.0172       0.8997       0.9686     0.0622     0.2156      60
   pcm32     S3_opcond_LOCO 0.9358 0.8620 0.8913    0.9851      0.7389   0.8  0.1358      0.0    0.0000       0.8986       0.9678     0.0619     0.2159      60
   pcm32   S3_topology_LOTO 0.8025 0.5115 0.4703    0.9965      0.0264   0.8  0.0025      0.0    0.0000       0.7047       0.8981    -0.0033     0.0089      60


### 3.3 Differences between policies

- Paired over the same 3,600 segments; physical runs (H1 + H2) resampled
- Pairs: S1→S2, S2→S2b, S2b→S3 (LOCO), S2b→S3′ (LOTO)
- `d_excess`: difference in excess over the training-fold majority rule

In [6]:
pairs = [("S1_segment_random", "S2_recording_indep", "recording-level sharing"),
         ("S2_recording_indep", "S2b_run_indep", "other hydrophone of the same run"),
         ("S2b_run_indep", "S3_opcond_LOCO", "unseen flow x noise regime"),
         ("S2b_run_indep", "S3_topology_LOTO", "unseen pipe topology")]
diffs = []
for a, b, meaning in pairs:
    r = sfb.paired_cluster_bootstrap_metrics(y, preds[a][0], preds[b][0], S.run_id, n_boot=cfg.N_BOOT,
                                             base_a=preds[a][1], base_b=preds[b][1])
    diffs.append(dict(step=f"{a} -> {b}", meaning=meaning, **r))
    for mtr in ["acc", "bacc", "excess"]:
        rp.record(f"D3.oof.{a}->{b}.d_{mtr}", r[f"d_{mtr}"], "E5_oof_policy_differences.csv",
                  "R1-M3,R1-M8,R2-11", lo=r[f"d_{mtr}_lo"], hi=r[f"d_{mtr}_hi"], note="unit=physical run")
diffs = pd.DataFrame(diffs)
diffs.to_csv(cfg.RESULTS / "E5_oof_policy_differences.csv", index=False)
print(diffs.round(4).to_string(index=False))
for _, r in oof.iterrows():
    rp.record(f"D3.oof.{r.policy}.acc", r.acc, "E5_oof_policies.csv", "R1-M7,R1-M8",
              lo=r.run_boot_lo, hi=r.run_boot_hi, note=f"bacc={r.bacc:.4f}; unit=run")
    rp.record(f"D3.oof.{r.policy}.bacc", r.bacc, "E5_oof_policies.csv", "R1-M7,R2-16")

                                   step                          meaning  n_units  n_clusters   d_acc  d_acc_lo  d_acc_hi  d_acc_excludes_zero  d_bacc  d_bacc_lo  d_bacc_hi  d_bacc_excludes_zero  d_excess  d_excess_lo  d_excess_hi  d_excess_excludes_zero
S1_segment_random -> S2_recording_indep          recording-level sharing     3600          60  0.0606    0.0294    0.0964                 True  0.1280     0.0742     0.1879                  True    0.0606       0.0294       0.0964                    True
    S2_recording_indep -> S2b_run_indep other hydrophone of the same run     3600          60 -0.0092   -0.0258    0.0064                False -0.0135    -0.0407     0.0145                 False   -0.0092      -0.0258       0.0064                   False
        S2b_run_indep -> S3_opcond_LOCO       unseen flow x noise regime     3600          60  0.0000   -0.0142    0.0142                False -0.0016    -0.0287     0.0273                 False    0.0000      -0.0142       0.0142     

## 4. Part C: int16 vs. PCM_32 decoding

Parts A and B repeated on int16-decoded features, identical splits

In [7]:
oof_leg, preds_leg = oof_all(Xleg, "int16_legacy")
s5_leg = e5_leg.groupby("protocol").agg(acc=("acc", "mean"), bacc=("bacc", "mean"),
                                        base=("base", "mean"), n=("acc", "size")).reset_index()
side = pd.concat([
    s5.assign(decoding="pcm32", design="v1 70/30")[["decoding", "design", "protocol", "acc", "bacc", "base", "n"]],
    s5_leg.assign(decoding="int16_legacy", design="v1 70/30")[["decoding", "design", "protocol", "acc", "bacc", "base", "n"]],
    oof.rename(columns={"policy": "protocol", "rule": "base"}).assign(design="OOF", n=cfg.N_SEEDS)[
        ["decoding", "design", "protocol", "acc", "bacc", "base", "n"]],
    oof_leg.rename(columns={"policy": "protocol", "rule": "base"}).assign(design="OOF", n=cfg.N_SEEDS)[
        ["decoding", "design", "protocol", "acc", "bacc", "base", "n"]],
], ignore_index=True)
side.to_csv(cfg.RESULTS / "E5_decoding_side_by_side.csv", index=False)
print(side.pivot_table(index=["design", "protocol"], columns="decoding",
                       values=["acc", "bacc"]).round(4).to_string())
for _, r in side.iterrows():
    rp.record(f"D3.decoding.{r.design}.{r.protocol}.{r.decoding}.acc", r.acc,
              "E5_decoding_side_by_side.csv", "R1-minor(self-disclosure),R2-14")

                                        acc                 bacc        
decoding                       int16_legacy   pcm32 int16_legacy   pcm32
design   protocol                                                       
OOF      S1_segment_random           0.9894  0.9872       0.9762  0.9748
         S2_recording_indep          0.9086  0.9267       0.7892  0.8469
         S2b_run_indep               0.9097  0.9358       0.8087  0.8604
         S3_opcond_LOCO              0.9250  0.9358       0.8250  0.8620
         S3_topology_LOTO            0.8139  0.8025       0.5415  0.5115
v1 70/30 S1_segment_random           0.9867  0.9874       0.9696  0.9737
         S2_recording_indep          0.8806  0.9052       0.7436  0.7952
         S3_condition_indep_v1       0.5728  0.5950       0.4406  0.4489
